# LAB 2 – Bài 4: Transfer Learning với Pretrained ResNet-50

## Phân loại món ăn VinaFood21 bằng Hugging Face Transformers

Trong các bài trước, mô hình được huấn luyện từ đầu. Ở bài này, chúng ta sử dụng một ResNet-50 đã được huấn luyện trước trên ImageNet và thích nghi mô hình cho 21 món ăn Việt Nam.

Notebook triển khai hai giai đoạn:

1. **Feature extraction:** đóng băng backbone, chỉ huấn luyện classifier mới.
2. **Fine-tuning:** mở tầng cuối của backbone và tiếp tục huấn luyện với learning rate nhỏ hơn.

## Mục tiêu học tập

Sau khi hoàn thành bài, sinh viên có thể:

1. Giải thích pretrained model, transfer learning và fine-tuning.
2. Dùng `AutoImageProcessor` và `ResNetForImageClassification`.
3. Thay classification head cho bài toán 21 lớp.
4. Đóng băng và mở lại từng phần của mô hình.
5. Hiểu vì sao BatchNorm cần được xử lý cẩn thận khi backbone bị đóng băng.
6. Tách train, validation và test đúng phương pháp.
7. Lưu checkpoint tốt nhất theo validation macro F1.
8. Đánh giá cuối cùng bằng test set đúng một lần.
9. Xuất mô hình theo định dạng Hugging Face để tái sử dụng.

> **Nguyên tắc quan trọng:** Không dùng test set để chọn epoch, learning rate hoặc chiến lược fine-tuning.

# 1. Cấu trúc dữ liệu

Dữ liệu cần có cấu trúc:

```text
VinaFood21/
├── train/
│   ├── Banh_chung/
│   ├── Banh_mi/
│   └── ...
└── test/
    ├── Banh_chung/
    ├── Banh_mi/
    └── ...
```

Mỗi thư mục con là một lớp.

Trên Kaggle, đường dẫn thường là:

```text
/kaggle/input/vinafood21/VinaFood21
```

Trên máy cá nhân, có thể đặt thư mục `VinaFood21` cùng cấp với notebook.

# 2. Cài đặt thư viện

Chỉ chạy khi môi trường chưa có thư viện:

```python
%pip install torch torchvision transformers scikit-learn matplotlib pillow
```

Lần đầu tải checkpoint `microsoft/resnet-50`, môi trường cần kết nối Internet. Sau đó mô hình có thể được đọc từ cache.

In [ ]:
# Bỏ dấu # ở dòng dưới nếu cần cài thư viện.
# %pip install torch torchvision transformers scikit-learn matplotlib pillow

# 3. Import và thiết lập tính tái lập

Random seed ảnh hưởng đến:

- Cách chia train và validation.
- Thứ tự batch.
- Data augmentation.
- Khởi tạo classifier mới.
- Một số phép toán trên GPU.

In [ ]:
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from transformers import (
    AutoImageProcessor,
    ResNetForImageClassification,
)


SEED = 42


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Thiết bị:", device)
print("PyTorch:", torch.__version__)

# 4. Cấu hình thí nghiệm

Bài này dùng hai learning rate khác nhau:

## Giai đoạn 1 – Feature extraction

Backbone bị đóng băng. Classifier mới được học với learning rate tương đối lớn.

## Giai đoạn 2 – Fine-tuning

Tầng cuối của backbone được mở. Backbone dùng learning rate nhỏ hơn classifier để tránh phá hỏng nhanh các đặc trưng đã học trước.

In [ ]:
MODEL_NAME = "microsoft/resnet-50"
NUM_CLASSES = 21

INPUT_SIZE = 224
RESIZE_SIZE = 256
BATCH_SIZE = 32

VALIDATION_RATIO = 0.15

FEATURE_EXTRACTION_EPOCHS = 5
FEATURE_EXTRACTION_LR = 1e-3

FINE_TUNING_EPOCHS = 10
BACKBONE_FINE_TUNING_LR = 1e-5
CLASSIFIER_FINE_TUNING_LR = 1e-4

WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 3

NUM_WORKERS = 2 if os.name != "nt" else 0
USE_MIXED_PRECISION = device.type == "cuda"

if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/pretrained_resnet_output")
else:
    OUTPUT_DIR = Path("./pretrained_resnet_output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "best_pretrained_resnet50.pth"
HISTORY_PATH = OUTPUT_DIR / "training_history.json"
TEST_RESULT_PATH = OUTPUT_DIR / "test_result.json"
EXPORT_DIR = OUTPUT_DIR / "huggingface_export"

print("Output directory:", OUTPUT_DIR.resolve())
print("Mixed precision:", USE_MIXED_PRECISION)

# 5. Tìm thư mục dữ liệu

Không nên viết cứng đường dẫn chỉ đúng trên một máy. Có thể sửa `MANUAL_DATA_ROOT` nếu dữ liệu nằm ở vị trí khác.

In [ ]:
# Ví dụ:
# MANUAL_DATA_ROOT = Path("/content/VinaFood21")
MANUAL_DATA_ROOT = None


def find_data_root(manual_root=None):
    candidates = []

    if manual_root is not None:
        candidates.append(Path(manual_root))

    candidates.extend(
        [
            Path("/kaggle/input/vinafood21/VinaFood21"),
            Path("/kaggle/input/vinafood21"),
            Path("./VinaFood21"),
            Path("./vinafood21/VinaFood21"),
            Path("./data/VinaFood21"),
            Path("./dataset/VinaFood21"),
            Path("../VinaFood21"),
        ]
    )

    for candidate in candidates:
        if (
            (candidate / "train").is_dir()
            and (candidate / "test").is_dir()
        ):
            return candidate

    checked_paths = "\n".join(
        f"- {path.resolve()}" for path in candidates
    )

    raise FileNotFoundError(
        "Không tìm thấy VinaFood21/train và VinaFood21/test.\n"
        "Các vị trí đã kiểm tra:\n"
        f"{checked_paths}\n\n"
        "Hãy sửa biến MANUAL_DATA_ROOT."
    )


DATA_ROOT = find_data_root(MANUAL_DATA_ROOT)
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

print("Data root:", DATA_ROOT.resolve())

# PHẦN A – LÝ THUYẾT TRANSFER LEARNING

# 6. Pretrained model là gì?

Pretrained model là mô hình đã được huấn luyện trước trên một tập dữ liệu lớn.

Các lớp đầu của CNN thường học đặc trưng tổng quát:

- Cạnh.
- Màu.
- Texture.
- Hình dạng đơn giản.

Các lớp sâu hơn học đặc trưng chuyên biệt hơn.

Khi chuyển sang VinaFood21, ta tận dụng các đặc trưng có sẵn thay vì khởi tạo toàn bộ mô hình ngẫu nhiên.

# 7. Feature extraction và fine-tuning

## Feature extraction

```text
Pretrained backbone: đóng băng
Classifier mới: được huấn luyện
```

Ưu điểm:

- Nhanh.
- Ít tốn GPU.
- Giảm nguy cơ overfitting khi dữ liệu nhỏ.

## Fine-tuning

```text
Một phần backbone: mở lại
Classifier: tiếp tục huấn luyện
```

Fine-tuning giúp đặc trưng thích nghi tốt hơn với món ăn Việt Nam, nhưng cần learning rate nhỏ.

# 8. Vì sao không fine-tune toàn bộ ngay từ đầu?

Classifier mới có trọng số ngẫu nhiên. Ở các batch đầu, gradient từ classifier có thể rất nhiễu.

Nếu mở toàn bộ backbone ngay và dùng learning rate lớn:

- Các đặc trưng pretrained có thể bị thay đổi quá nhanh.
- Mô hình có thể quên kiến thức đã học trước.
- Huấn luyện kém ổn định.

Vì vậy, quy trình an toàn là học classifier trước rồi mới mở dần backbone.

# 9. Catastrophic forgetting

Catastrophic forgetting xảy ra khi fine-tuning làm mô hình mất nhanh các biểu diễn hữu ích đã học trước.

Cách hạn chế:

- Learning rate nhỏ cho backbone.
- Mở dần từng stage.
- Early stopping.
- Weight decay.
- Theo dõi validation set.

# PHẦN B – DỮ LIỆU VÀ PREPROCESSING

# 10. Tải image processor của checkpoint

`AutoImageProcessor` cung cấp thông tin tiền xử lý đi kèm checkpoint, gồm mean và standard deviation.

Việc dùng đúng thống kê tiền xử lý rất quan trọng vì backbone pretrained đã học trên dữ liệu được chuẩn hóa theo cách đó.

In [ ]:
try:
    image_processor = AutoImageProcessor.from_pretrained(
        MODEL_NAME
    )
except Exception as error:
    raise RuntimeError(
        "Không tải được image processor. "
        "Hãy bật Internet hoặc kiểm tra cache Hugging Face."
    ) from error

IMAGE_MEAN = list(image_processor.image_mean)
IMAGE_STD = list(image_processor.image_std)

print("Image mean:", IMAGE_MEAN)
print("Image std:", IMAGE_STD)
print("Processor size:", image_processor.size)

# 11. Transform cho train và validation/test

Train có augmentation ngẫu nhiên để giảm overfitting.

Validation và test dùng transform xác định để kết quả ổn định.

In [ ]:
normalize = transforms.Normalize(
    mean=IMAGE_MEAN,
    std=IMAGE_STD,
)

train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(
            INPUT_SIZE,
            scale=(0.75, 1.0),
            ratio=(0.90, 1.10),
        ),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.15,
        ),
        transforms.ToTensor(),
        normalize,
    ]
)

evaluation_transform = transforms.Compose(
    [
        transforms.Resize(RESIZE_SIZE),
        transforms.CenterCrop(INPUT_SIZE),
        transforms.ToTensor(),
        normalize,
    ]
)

print(train_transform)

# 12. Đọc dữ liệu bằng ImageFolder

`ImageFolder` tự động đọc cấu trúc thư mục lớp và trả về `(image, label)`.

Không cần tự viết Dataset và `collate_fn` cho trường hợp này.

In [ ]:
train_full_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=train_transform,
)

validation_full_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=evaluation_transform,
)

test_dataset = datasets.ImageFolder(
    root=TEST_DIR,
    transform=evaluation_transform,
)

print("Số ảnh train ban đầu:", len(train_full_dataset))
print("Số ảnh test:", len(test_dataset))
print("Tên lớp:", train_full_dataset.classes)

# 13. Kiểm tra class mapping

Train và test phải dùng cùng ánh xạ tên lớp sang chỉ số.

In [ ]:
if train_full_dataset.class_to_idx != test_dataset.class_to_idx:
    raise ValueError(
        "class_to_idx của train và test không giống nhau."
    )

if len(train_full_dataset.classes) != NUM_CLASSES:
    raise ValueError(
        f"NUM_CLASSES={NUM_CLASSES}, nhưng dữ liệu có "
        f"{len(train_full_dataset.classes)} lớp."
    )

if train_full_dataset.samples != validation_full_dataset.samples:
    raise ValueError(
        "Thứ tự mẫu giữa train dataset và validation dataset "
        "không giống nhau."
    )

class_names = train_full_dataset.classes
class_to_idx = train_full_dataset.class_to_idx

id2label = {
    index: class_name
    for index, class_name in enumerate(class_names)
}

label2id = {
    class_name: index
    for index, class_name in enumerate(class_names)
}

print(class_to_idx)

# 14. Tách validation có phân tầng

Validation được lấy từ thư mục train.

`stratify` giúp giữ tỷ lệ lớp gần giống nhau giữa hai tập.

In [ ]:
all_indices = np.arange(len(train_full_dataset))
all_targets = np.asarray(train_full_dataset.targets)

class_counts = np.bincount(
    all_targets,
    minlength=NUM_CLASSES,
)

if class_counts.min() < 2:
    raise ValueError(
        "Mỗi lớp cần ít nhất 2 ảnh để chia có phân tầng."
    )

train_indices, validation_indices = train_test_split(
    all_indices,
    test_size=VALIDATION_RATIO,
    random_state=SEED,
    shuffle=True,
    stratify=all_targets,
)

train_dataset = Subset(
    train_full_dataset,
    train_indices,
)

validation_dataset = Subset(
    validation_full_dataset,
    validation_indices,
)

print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))

# 15. Khảo sát phân bố lớp

In [ ]:
train_targets = all_targets[train_indices]
validation_targets = all_targets[validation_indices]

train_counts = np.bincount(
    train_targets,
    minlength=NUM_CLASSES,
)

validation_counts = np.bincount(
    validation_targets,
    minlength=NUM_CLASSES,
)

for class_index, class_name in enumerate(class_names):
    print(
        f"{class_index:02d} {class_name:25s} | "
        f"train: {train_counts[class_index]:4d} | "
        f"validation: {validation_counts[class_index]:4d}"
    )

In [ ]:
positions = np.arange(NUM_CLASSES)

plt.figure(figsize=(14, 5))
plt.bar(positions, train_counts)
plt.xticks(positions, class_names, rotation=90)
plt.xlabel("Lớp")
plt.ylabel("Số ảnh")
plt.title("Phân bố lớp trong train set")
plt.tight_layout()
plt.show()

# 16. DataLoader

In [ ]:
loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

pin_memory = device.type == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker,
)

first_images, first_labels = next(iter(train_loader))

print("Batch ảnh:", first_images.shape)
print("Batch nhãn:", first_labels.shape)

# 17. Hiển thị ảnh sau khi bỏ Normalize

In [ ]:
def denormalize_image(image_tensor):
    mean = torch.tensor(IMAGE_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGE_STD).view(3, 1, 1)

    image = image_tensor.cpu() * std + mean
    return image.clamp(0, 1)


number_of_samples = min(8, len(first_images))

figure, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for index in range(number_of_samples):
    image = denormalize_image(first_images[index])
    image = image.permute(1, 2, 0)

    axes[index].imshow(image)
    axes[index].set_title(
        class_names[first_labels[index].item()]
    )
    axes[index].axis("off")

for index in range(number_of_samples, len(axes)):
    axes[index].axis("off")

plt.tight_layout()
plt.show()

# PHẦN C – KHỞI TẠO PRETRAINED MODEL

# 18. Tải ResNet-50 pretrained và thay classifier

`ignore_mismatched_sizes=True` cho phép tải backbone pretrained dù classifier ImageNet có số lớp khác 21.

Classification head mới được khởi tạo ngẫu nhiên.

In [ ]:
try:
    model = ResNetForImageClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_CLASSES,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )
except Exception as error:
    raise RuntimeError(
        "Không tải được pretrained model. "
        "Hãy bật Internet hoặc kiểm tra cache Hugging Face."
    ) from error

model = model.to(device)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"Tổng tham số: {total_parameters:,}")
print(f"Tham số đang trainable: {trainable_parameters:,}")

# 19. Kiểm tra output

Hugging Face trả về một đối tượng có thuộc tính `.logits`.

Với batch `(B, 3, 224, 224)`, logits phải có shape `(B, 21)`.

In [ ]:
model.eval()

dummy_images = torch.randn(
    2,
    3,
    INPUT_SIZE,
    INPUT_SIZE,
    device=device,
)

with torch.inference_mode():
    dummy_outputs = model(
        pixel_values=dummy_images
    )
    dummy_logits = dummy_outputs.logits

print("Input:", dummy_images.shape)
print("Logits:", dummy_logits.shape)

assert dummy_logits.shape == (2, NUM_CLASSES)

del dummy_images, dummy_outputs, dummy_logits

if device.type == "cuda":
    torch.cuda.empty_cache()

# 20. Xem tên các tham số chính

Việc xem tên tham số giúp hiểu phần nào thuộc backbone và phần nào thuộc classifier.

In [ ]:
for name, parameter in list(
    model.named_parameters()
)[-12:]:
    print(
        f"{name:65s} "
        f"shape={tuple(parameter.shape)}"
    )

# PHẦN D – ĐÓNG BĂNG VÀ MỞ BACKBONE

# 21. Giai đoạn 1: chỉ huấn luyện classifier

Tất cả tham số backbone được đặt `requires_grad=False`.

Chỉ classifier nhận gradient và được optimizer cập nhật.

In [ ]:
def freeze_backbone_train_classifier(model):
    for parameter in model.resnet.parameters():
        parameter.requires_grad = False

    for parameter in model.classifier.parameters():
        parameter.requires_grad = True


freeze_backbone_train_classifier(model)

trainable_names = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

print("Các tham số trainable:")
for name in trainable_names:
    print("-", name)

print(
    "Số tham số trainable:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}",
)

# 22. Vì sao BatchNorm cần chú ý?

`requires_grad=False` chỉ ngăn weight và bias được cập nhật bởi gradient.

Khi mô hình ở `train()`:

- BatchNorm vẫn có thể cập nhật running mean và running variance.
- Dropout vẫn hoạt động ngẫu nhiên.

Với các module đang đóng băng, ta đặt BatchNorm về `eval()` để giữ nguyên thống kê pretrained.

In [ ]:
def set_frozen_batchnorm_to_eval(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module_parameters = list(
                module.parameters(recurse=False)
            )

            is_frozen = (
                len(module_parameters) > 0
                and all(
                    not parameter.requires_grad
                    for parameter in module_parameters
                )
            )

            if is_frozen:
                module.eval()

# 23. Giai đoạn 2: mở stage cuối

ResNet-50 có nhiều stage. Ta chỉ mở stage cuối để:

- Thích nghi đặc trưng cấp cao với món ăn Việt Nam.
- Giữ phần lớn đặc trưng tổng quát.
- Giảm chi phí tính toán.
- Hạn chế catastrophic forgetting.

In [ ]:
def unfreeze_last_resnet_stage(model):
    for parameter in model.resnet.parameters():
        parameter.requires_grad = False

    for parameter in model.classifier.parameters():
        parameter.requires_grad = True

    last_stage = model.resnet.encoder.stages[-1]

    for parameter in last_stage.parameters():
        parameter.requires_grad = True

    return last_stage

# PHẦN E – LOSS, METRICS VÀ TRAINING FUNCTIONS

# 24. CrossEntropyLoss

Mô hình trả về logits. Không thêm Softmax trước loss.

In [ ]:
loss_fn = nn.CrossEntropyLoss()


def create_grad_scaler():
    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=USE_MIXED_PRECISION,
        )
    except TypeError:
        return torch.cuda.amp.GradScaler(
            enabled=USE_MIXED_PRECISION,
        )


scaler = create_grad_scaler()

# 25. Hàm đánh giá

Các chỉ số gồm:

- Loss.
- Accuracy.
- Macro precision.
- Macro recall.
- Macro F1.
- Top-3 accuracy.

In [ ]:
def evaluate_model(
    model,
    dataloader,
    loss_fn,
    return_predictions=False,
):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    total_top3_correct = 0

    all_labels = []
    all_predictions = []

    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(
                device,
                non_blocking=True,
            )
            labels = labels.to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type=device.type,
                enabled=USE_MIXED_PRECISION,
            ):
                outputs = model(
                    pixel_values=images
                )
                logits = outputs.logits
                loss = loss_fn(logits, labels)

            predictions = logits.argmax(dim=1)
            top3_indices = logits.topk(
                k=min(3, NUM_CLASSES),
                dim=1,
            ).indices

            batch_size = images.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            total_top3_correct += (
                top3_indices
                .eq(labels.unsqueeze(1))
                .any(dim=1)
                .sum()
                .item()
            )

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(
                predictions.cpu().numpy()
            )

    y_true = np.asarray(all_labels)
    y_pred = np.asarray(all_predictions)

    metrics = {
        "loss": total_loss / total_samples,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "top3_accuracy": (
            total_top3_correct / total_samples
        ),
    }

    if return_predictions:
        metrics["y_true"] = y_true
        metrics["y_pred"] = y_pred

    return metrics

# 26. Hàm huấn luyện một epoch

Sau `model.train()`, hàm đặt các BatchNorm thuộc phần đóng băng về lại chế độ eval.

In [ ]:
def train_one_epoch(
    model,
    dataloader,
    loss_fn,
    optimizer,
    scaler,
):
    model.train()
    set_frozen_batchnorm_to_eval(model)

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in dataloader:
        images = images.to(
            device,
            non_blocking=True,
        )
        labels = labels.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            enabled=USE_MIXED_PRECISION,
        ):
            outputs = model(
                pixel_values=images
            )
            logits = outputs.logits
            loss = loss_fn(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        predictions = logits.argmax(dim=1)
        batch_size = images.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (
            predictions == labels
        ).sum().item()
        total_samples += batch_size

    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples,
    }

# 27. Hàm chạy một giai đoạn huấn luyện

Checkpoint tốt nhất được chọn bằng validation macro F1 trên cả hai giai đoạn.

In [ ]:
history = {
    "phase": [],
    "global_epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "validation_loss": [],
    "validation_accuracy": [],
    "validation_f1_macro": [],
    "validation_top3_accuracy": [],
    "classifier_lr": [],
}

best_validation_f1 = float("-inf")
global_epoch = 0


def run_training_phase(
    phase_name,
    model,
    optimizer,
    scheduler,
    number_of_epochs,
    patience,
    global_epoch_start,
    best_f1_start,
):
    global_epoch = global_epoch_start
    best_f1 = best_f1_start
    epochs_without_improvement = 0

    for local_epoch in range(
        1,
        number_of_epochs + 1,
    ):
        global_epoch += 1

        train_metrics = train_one_epoch(
            model=model,
            dataloader=train_loader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            scaler=scaler,
        )

        validation_metrics = evaluate_model(
            model=model,
            dataloader=validation_loader,
            loss_fn=loss_fn,
        )

        scheduler.step(validation_metrics["loss"])

        classifier_lr = optimizer.param_groups[-1]["lr"]

        history["phase"].append(phase_name)
        history["global_epoch"].append(global_epoch)
        history["train_loss"].append(
            float(train_metrics["loss"])
        )
        history["train_accuracy"].append(
            float(train_metrics["accuracy"])
        )
        history["validation_loss"].append(
            float(validation_metrics["loss"])
        )
        history["validation_accuracy"].append(
            float(validation_metrics["accuracy"])
        )
        history["validation_f1_macro"].append(
            float(validation_metrics["f1_macro"])
        )
        history["validation_top3_accuracy"].append(
            float(validation_metrics["top3_accuracy"])
        )
        history["classifier_lr"].append(
            float(classifier_lr)
        )

        print(
            f"[{phase_name}] "
            f"Epoch {local_epoch:02d}/{number_of_epochs} | "
            f"Global {global_epoch:02d} | "
            f"Train loss: {train_metrics['loss']:.4f} | "
            f"Train acc: {train_metrics['accuracy']:.4f} | "
            f"Val loss: {validation_metrics['loss']:.4f} | "
            f"Val acc: {validation_metrics['accuracy']:.4f} | "
            f"Val macro F1: {validation_metrics['f1_macro']:.4f} | "
            f"Val top-3: {validation_metrics['top3_accuracy']:.4f}"
        )

        if validation_metrics["f1_macro"] > best_f1:
            best_f1 = validation_metrics["f1_macro"]
            epochs_without_improvement = 0

            checkpoint = {
                "phase": phase_name,
                "global_epoch": global_epoch,
                "local_epoch": local_epoch,
                "model_name": MODEL_NAME,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),
                "scheduler_state_dict": (
                    scheduler.state_dict()
                ),
                "validation_metrics": {
                    key: float(value)
                    for key, value
                    in validation_metrics.items()
                },
                "class_names": class_names,
                "id2label": id2label,
                "label2id": label2id,
                "config": {
                    "seed": SEED,
                    "num_classes": NUM_CLASSES,
                    "input_size": INPUT_SIZE,
                    "batch_size": BATCH_SIZE,
                    "validation_ratio": VALIDATION_RATIO,
                    "weight_decay": WEIGHT_DECAY,
                },
            }

            torch.save(
                checkpoint,
                CHECKPOINT_PATH,
            )

            print(
                "  -> Đã lưu checkpoint mới. "
                f"Validation macro F1 = {best_f1:.4f}"
            )
        else:
            epochs_without_improvement += 1

            print(
                "  -> Chưa cải thiện. "
                f"Early stopping: "
                f"{epochs_without_improvement}/{patience}"
            )

        if epochs_without_improvement >= patience:
            print(
                f"Dừng sớm giai đoạn {phase_name}."
            )
            break

    return global_epoch, best_f1

# PHẦN F – GIAI ĐOẠN 1: FEATURE EXTRACTION

# 28. Tạo optimizer chỉ cho classifier

Optimizer chỉ nhận các tham số có `requires_grad=True`.

In [ ]:
freeze_backbone_train_classifier(model)

feature_optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters(),
    ),
    lr=FEATURE_EXTRACTION_LR,
    weight_decay=WEIGHT_DECAY,
)

feature_scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        feature_optimizer,
        mode="min",
        factor=0.5,
        patience=1,
    )
)

print(
    "Tham số trainable giai đoạn 1:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}",
)

# 29. Huấn luyện classifier

In [ ]:
global_epoch, best_validation_f1 = run_training_phase(
    phase_name="feature_extraction",
    model=model,
    optimizer=feature_optimizer,
    scheduler=feature_scheduler,
    number_of_epochs=FEATURE_EXTRACTION_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    global_epoch_start=global_epoch,
    best_f1_start=best_validation_f1,
)

# PHẦN G – GIAI ĐOẠN 2: FINE-TUNING

# 30. Tải lại checkpoint tốt nhất trước khi mở backbone

Fine-tuning nên bắt đầu từ phiên bản tốt nhất của giai đoạn trước, không nhất thiết từ epoch cuối.

In [ ]:
phase_one_checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
)

model.load_state_dict(
    phase_one_checkpoint["model_state_dict"]
)

print(
    "Đã tải checkpoint:",
    phase_one_checkpoint["phase"],
    "global epoch",
    phase_one_checkpoint["global_epoch"],
)

# 31. Mở stage cuối và dùng discriminative learning rates

- Stage cuối: learning rate rất nhỏ.
- Classifier: learning rate lớn hơn.

Đây gọi là **discriminative learning rates**.

In [ ]:
last_stage = unfreeze_last_resnet_stage(model)

fine_tuning_optimizer = torch.optim.AdamW(
    [
        {
            "params": last_stage.parameters(),
            "lr": BACKBONE_FINE_TUNING_LR,
        },
        {
            "params": model.classifier.parameters(),
            "lr": CLASSIFIER_FINE_TUNING_LR,
        },
    ],
    weight_decay=WEIGHT_DECAY,
)

fine_tuning_scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        fine_tuning_optimizer,
        mode="min",
        factor=0.5,
        patience=1,
    )
)

print(
    "Tham số trainable giai đoạn 2:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}",
)
print(
    "Backbone LR:",
    fine_tuning_optimizer.param_groups[0]["lr"],
)
print(
    "Classifier LR:",
    fine_tuning_optimizer.param_groups[1]["lr"],
)

# 32. Fine-tune

In [ ]:
global_epoch, best_validation_f1 = run_training_phase(
    phase_name="fine_tuning",
    model=model,
    optimizer=fine_tuning_optimizer,
    scheduler=fine_tuning_scheduler,
    number_of_epochs=FINE_TUNING_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    global_epoch_start=global_epoch,
    best_f1_start=best_validation_f1,
)

with HISTORY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        history,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Đã lưu history:", HISTORY_PATH.resolve())
print("Checkpoint tốt nhất:", CHECKPOINT_PATH.resolve())

# PHẦN H – BIỂU ĐỒ HUẤN LUYỆN

Đường phân cách cho biết thời điểm chuyển từ feature extraction sang fine-tuning.

In [ ]:
epochs = history["global_epoch"]

phase_change_positions = [
    index + 1
    for index in range(1, len(history["phase"]))
    if history["phase"][index]
    != history["phase"][index - 1]
]

plt.figure(figsize=(10, 5))
plt.plot(
    epochs,
    history["train_loss"],
    marker="o",
    label="Train loss",
)
plt.plot(
    epochs,
    history["validation_loss"],
    marker="o",
    label="Validation loss",
)

for position in phase_change_positions:
    plt.axvline(
        x=position,
        linestyle="--",
        label="Bắt đầu fine-tuning",
    )

plt.xlabel("Global epoch")
plt.ylabel("Loss")
plt.title("Loss qua hai giai đoạn")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    epochs,
    history["validation_accuracy"],
    marker="o",
    label="Validation accuracy",
)
plt.plot(
    epochs,
    history["validation_f1_macro"],
    marker="o",
    label="Validation macro F1",
)
plt.plot(
    epochs,
    history["validation_top3_accuracy"],
    marker="o",
    label="Validation top-3 accuracy",
)

for position in phase_change_positions:
    plt.axvline(
        x=position,
        linestyle="--",
        label="Bắt đầu fine-tuning",
    )

plt.xlabel("Global epoch")
plt.ylabel("Score")
plt.title("Validation metrics")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.show()

# PHẦN I – ĐÁNH GIÁ CUỐI CÙNG

# 33. Tải checkpoint tốt nhất

Checkpoint tốt nhất có thể thuộc feature extraction hoặc fine-tuning.

In [ ]:
best_checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

print("Best phase:", best_checkpoint["phase"])
print("Best global epoch:", best_checkpoint["global_epoch"])
print(
    "Best validation macro F1:",
    f"{best_checkpoint['validation_metrics']['f1_macro']:.4f}",
)

# 34. Chỉ bây giờ mới dùng test set

In [ ]:
test_metrics = evaluate_model(
    model=model,
    dataloader=test_loader,
    loss_fn=loss_fn,
    return_predictions=True,
)

print("KẾT QUẢ CUỐI CÙNG TRÊN TEST SET")
print("-" * 50)
print(f"Loss:            {test_metrics['loss']:.4f}")
print(f"Accuracy:        {test_metrics['accuracy']:.4f}")
print(f"Macro precision: {test_metrics['precision_macro']:.4f}")
print(f"Macro recall:    {test_metrics['recall_macro']:.4f}")
print(f"Macro F1:        {test_metrics['f1_macro']:.4f}")
print(f"Top-3 accuracy:  {test_metrics['top3_accuracy']:.4f}")

In [ ]:
serializable_test_result = {
    key: float(value)
    for key, value in test_metrics.items()
    if key not in {"y_true", "y_pred"}
}

with TEST_RESULT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        serializable_test_result,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Đã lưu test result:", TEST_RESULT_PATH.resolve())

# 35. Classification report

In [ ]:
print(
    classification_report(
        test_metrics["y_true"],
        test_metrics["y_pred"],
        target_names=class_names,
        digits=4,
        zero_division=0,
    )
)

# 36. Confusion matrix chuẩn hóa

In [ ]:
normalized_matrix = confusion_matrix(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    normalize="true",
)

display = ConfusionMatrixDisplay(
    confusion_matrix=normalized_matrix,
    display_labels=class_names,
)

figure, axis = plt.subplots(figsize=(16, 16))
display.plot(
    ax=axis,
    values_format=".2f",
    xticks_rotation=90,
)
axis.set_title("Normalized confusion matrix")
plt.tight_layout()
plt.show()

# 37. Accuracy theo từng lớp và lớp khó nhất

In [ ]:
per_class_accuracy = np.diag(normalized_matrix)
positions = np.arange(NUM_CLASSES)

plt.figure(figsize=(14, 5))
plt.bar(positions, per_class_accuracy)
plt.xticks(positions, class_names, rotation=90)
plt.ylim(0, 1)
plt.xlabel("Lớp")
plt.ylabel("Accuracy theo lớp")
plt.title("Độ chính xác của từng lớp")
plt.tight_layout()
plt.show()

hardest_class_index = int(
    np.argmin(per_class_accuracy)
)

print(
    "Lớp khó nhất:",
    class_names[hardest_class_index],
)
print(
    "Accuracy:",
    f"{per_class_accuracy[hardest_class_index]:.4f}",
)

# 38. Các cặp lớp bị nhầm nhiều nhất

In [ ]:
raw_matrix = confusion_matrix(
    test_metrics["y_true"],
    test_metrics["y_pred"],
)

confusion_without_diagonal = raw_matrix.copy()
np.fill_diagonal(confusion_without_diagonal, 0)

flat_indices = np.argsort(
    confusion_without_diagonal.ravel()
)[::-1]

print("Các cặp lớp bị nhầm nhiều nhất:")

number_printed = 0

for flat_index in flat_indices:
    true_index, predicted_index = np.unravel_index(
        flat_index,
        confusion_without_diagonal.shape,
    )

    count = confusion_without_diagonal[
        true_index,
        predicted_index,
    ]

    if count == 0:
        break

    print(
        f"- Thật: {class_names[true_index]:25s} "
        f"→ Đoán: {class_names[predicted_index]:25s} "
        f"| {count} ảnh"
    )

    number_printed += 1

    if number_printed >= 10:
        break

# 39. Hiển thị ảnh dự đoán sai

In [ ]:
def collect_misclassified_samples(
    model,
    dataloader,
    max_samples=12,
):
    model.eval()
    samples = []

    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            with torch.autocast(
                device_type=device.type,
                enabled=USE_MIXED_PRECISION,
            ):
                logits = model(
                    pixel_values=images
                ).logits

            probabilities = torch.softmax(
                logits,
                dim=1,
            )
            predictions = logits.argmax(dim=1)

            wrong_indices = (
                predictions != labels
            ).nonzero(as_tuple=True)[0]

            for index in wrong_indices:
                predicted_index = (
                    predictions[index].item()
                )

                samples.append(
                    {
                        "image": images[index].cpu(),
                        "true_index": labels[index].item(),
                        "predicted_index": predicted_index,
                        "confidence": probabilities[
                            index,
                            predicted_index,
                        ].item(),
                    }
                )

                if len(samples) >= max_samples:
                    return samples

    return samples


misclassified_samples = collect_misclassified_samples(
    model=model,
    dataloader=test_loader,
    max_samples=12,
)

print("Số ảnh sai:", len(misclassified_samples))

In [ ]:
if misclassified_samples:
    figure, axes = plt.subplots(
        3,
        4,
        figsize=(15, 11),
    )
    axes = axes.flatten()

    for axis, sample in zip(
        axes,
        misclassified_samples,
    ):
        image = denormalize_image(
            sample["image"]
        )
        image = image.permute(1, 2, 0)

        true_name = class_names[
            sample["true_index"]
        ]
        predicted_name = class_names[
            sample["predicted_index"]
        ]

        axis.imshow(image)
        axis.set_title(
            f"Thật: {true_name}\n"
            f"Đoán: {predicted_name}\n"
            f"Tự tin: {sample['confidence']:.2%}",
            fontsize=9,
        )
        axis.axis("off")

    for axis in axes[len(misclassified_samples):]:
        axis.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Không tìm thấy ảnh dự đoán sai.")

# PHẦN J – XUẤT MÔ HÌNH VÀ DỰ ĐOÁN

# 40. Xuất mô hình theo định dạng Hugging Face

Sau khi tải checkpoint tốt nhất, lưu model và image processor vào cùng một thư mục.

Thư mục này có thể được tải lại bằng `from_pretrained()`.

In [ ]:
EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

model.save_pretrained(EXPORT_DIR)
image_processor.save_pretrained(EXPORT_DIR)

print("Đã xuất mô hình tại:", EXPORT_DIR.resolve())

# 41. Tải lại mô hình đã xuất

Đây là sanity check quan trọng trước khi triển khai.

In [ ]:
reloaded_processor = AutoImageProcessor.from_pretrained(
    EXPORT_DIR
)

reloaded_model = (
    ResNetForImageClassification.from_pretrained(
        EXPORT_DIR
    )
    .to(device)
)

reloaded_model.eval()

print("Đã tải lại mô hình thành công.")

# 42. Dự đoán một ảnh bên ngoài

Hàm trả về ba lớp có xác suất cao nhất.

In [ ]:
def predict_image(
    model,
    image_path,
    transform,
    class_names,
    top_k=3,
):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    image_tensor = (
        transform(image)
        .unsqueeze(0)
        .to(device)
    )

    with torch.inference_mode():
        with torch.autocast(
            device_type=device.type,
            enabled=USE_MIXED_PRECISION,
        ):
            logits = model(
                pixel_values=image_tensor
            ).logits

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        top_probabilities, top_indices = (
            probabilities.topk(
                k=min(top_k, len(class_names)),
                dim=1,
            )
        )

    results = []

    for probability, class_index in zip(
        top_probabilities[0].cpu(),
        top_indices[0].cpu(),
    ):
        results.append(
            {
                "class_name": (
                    class_names[class_index.item()]
                ),
                "probability": probability.item(),
            }
        )

    return image, results


# Ví dụ:
# image_path = TEST_DIR / class_names[0] / "ten_anh.jpg"
#
# image, results = predict_image(
#     model=reloaded_model,
#     image_path=image_path,
#     transform=evaluation_transform,
#     class_names=class_names,
# )
#
# plt.figure(figsize=(6, 6))
# plt.imshow(image)
# plt.axis("off")
# plt.show()
#
# for result in results:
#     print(
#         result["class_name"],
#         f"{result['probability']:.2%}",
#     )

# PHẦN K – TỔNG KẾT

# 43. Quy trình chuẩn

```text
1. Đọc metadata từ image processor
2. Tạo train và evaluation transforms
3. Đọc dữ liệu bằng ImageFolder
4. Kiểm tra class mapping
5. Tách validation có phân tầng
6. Tải pretrained model
7. Thay classifier 21 lớp
8. Đóng băng backbone
9. Huấn luyện classifier
10. Tải checkpoint tốt nhất
11. Mở stage cuối
12. Fine-tune với learning rate nhỏ
13. Tải checkpoint tốt nhất toàn bộ
14. Test đúng một lần
15. Phân tích lỗi
16. Xuất model và processor
```

# 44. Các lỗi thường gặp

## Lỗi 1: Dùng test set sau mỗi epoch

Test bị rò rỉ vào quá trình lựa chọn mô hình.

## Lỗi 2: Fine-tune toàn bộ ngay từ đầu với learning rate lớn

Có thể phá hỏng nhanh các đặc trưng pretrained.

## Lỗi 3: Đóng băng weight nhưng để BatchNorm cập nhật statistics

Backbone không thật sự được giữ nguyên.

## Lỗi 4: Tự viết Dataset dù ImageFolder đã đáp ứng

Code dài hơn và dễ bỏ sót định dạng ảnh.

## Lỗi 5: Không dùng preprocessing phù hợp checkpoint

Phân phối đầu vào khác với lúc pretrained.

## Lỗi 6: Dùng cùng learning rate cho backbone và classifier

Backbone thường cần learning rate nhỏ hơn.

## Lỗi 7: Chỉ lưu state_dict

Khó tái sử dụng nếu thiếu tên lớp và processor.

## Lỗi 8: Không tải lại checkpoint tốt nhất

Epoch cuối chưa chắc tốt nhất.

## Lỗi 9: Chỉ báo accuracy

Có thể che giấu lớp có chất lượng thấp.

# 45. Câu hỏi tự kiểm tra

1. Pretrained model mang lại lợi ích gì?
2. Feature extraction khác fine-tuning như thế nào?
3. Vì sao phải học classifier trước?
4. Catastrophic forgetting là gì?
5. Vì sao backbone dùng learning rate nhỏ?
6. `requires_grad=False` có tự dừng BatchNorm statistics không?
7. Vì sao cần dùng image processor của checkpoint?
8. Validation set có vai trò gì?
9. Tại sao test chỉ dùng một lần?
10. Khi nào nên mở thêm nhiều stage?

# 46. Bài tập mở rộng

## Bài 1 – Chỉ feature extraction

Không fine-tune backbone. So sánh validation macro F1.

## Bài 2 – Mở nhiều stage

So sánh mở 1 stage cuối và 2 stage cuối.

## Bài 3 – Discriminative learning rates

Thử các tỷ lệ learning rate giữa backbone và classifier.

## Bài 4 – So sánh pretrained và training from scratch

Dùng cùng train/validation split để so sánh công bằng.

## Bài 5 – Data augmentation

So sánh augmentation nhẹ và mạnh.

## Bài 6 – Xử lý mất cân bằng

Thử class weights hoặc WeightedRandomSampler.

## Bài 7 – Phân tích lỗi

Chọn ba cặp món ăn bị nhầm nhiều nhất và giải thích bằng ảnh.

## Bài 8 – Báo cáo

Báo cáo gồm:

1. Cấu hình pretrained checkpoint.
2. Chiến lược freeze và unfreeze.
3. Learning rate từng nhóm tham số.
4. Biểu đồ hai giai đoạn.
5. Validation tốt nhất.
6. Test cuối cùng.
7. Confusion matrix.
8. Ảnh sai.
9. So sánh với ResNet huấn luyện từ đầu.

# Checklist trước khi nộp

- [ ] Đã chạy `Restart & Run All`.
- [ ] Không có ô code báo lỗi.
- [ ] Có đúng 21 lớp.
- [ ] Train và test có cùng class mapping.
- [ ] Validation không dùng augmentation ngẫu nhiên.
- [ ] Không dùng test để chọn mô hình.
- [ ] Đã huấn luyện classifier trước.
- [ ] Backbone dùng learning rate nhỏ khi fine-tune.
- [ ] BatchNorm đóng băng không cập nhật statistics.
- [ ] Có checkpoint tốt nhất.
- [ ] Có biểu đồ hai giai đoạn.
- [ ] Có classification report.
- [ ] Có confusion matrix.
- [ ] Có ảnh dự đoán sai.
- [ ] Đã xuất model và image processor.
- [ ] Đã lưu notebook sau khi chạy.

# Tài liệu chính thức nên đọc thêm

- Hugging Face Transformers: ResNet.
- Hugging Face Transformers: AutoImageProcessor.
- Torchvision: ImageFolder.
- PyTorch: Automatic Mixed Precision.